0. Imports.

In [31]:
import pandas as pd
from sqlalchemy import create_engine, VARCHAR, CHAR, TEXT, DateTime, Integer, Float

1. Loading data.

In [32]:
tables = {
    'customers': 'olist_order_customers_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

olist_df = {}
for name, archive in tables.items():
    df = pd.read_csv(f'../data/raw/{archive}')
    olist_df[name] = df

2. olist_orders_dataset.csv.

-2.1. Converting data types:
  * Converting columns from `order_purchase_timestamp` (str) to `datetime`;
  * Mapping the columns.

In [ ]:
columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
                    'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in columns:
    olist_df['orders'][col] = pd.to_datetime(olist_df['orders'][col])

dtype_orders = {
    'order_id': VARCHAR(32),
    'customer_id': VARCHAR(32),
    'order_status': VARCHAR(20),
    'order_purchase_timestamp': DateTime(),
    'order_approved_at': DateTime(),
    'order_delivered_carrier_date': DateTime(),
    'order_delivered_customer_date': DateTime(),
    'order_estimated_delivery_date': DateTime()
}

- 2.2. Creating columns:
  * delivery_delay_days: days of delay (int) in delivery, essential for business questions 1, 2, 3, 12 and 14;
  * order_processing_time_days: days between purchase and approval (int), useful for business question 5;
  * purchase_year_month: year and month of purchase (str), facilitates temporal aggregations in SQL for questions 1 and 6.

In [ ]:
olist_df['orders']['delivery_delay_days'] = (olist_df['orders']['order_delivered_customer_date'] - olist_df['orders']['order_estimated_delivery_date']).dt.days
olist_df['orders']['order_processing_time_days'] = (olist_df['orders']['order_approved_at'] - olist_df['orders']['order_purchase_timestamp']).dt.days
olist_df['orders']['purchase_year_month'] = olist_df['orders']['order_purchase_timestamp'].dt.to_period('M').astype(str)

- 2.3. Null values: Nulls in the columns 'order_approved_at', 'order_delivered_carrier_date', and 'order_delivered_customer_date' have logical meaning (cancelled, unapproved, or undelivered orders), therefore, the values ​​should be kept;
- 2.4. Column names: Already standardized;
- 2.5. Duplicate columns: None.

3. olist_products_dataset.

- 3.1. Null values:
  * product_category_name: remove null rows, as this is the focus of business questions 3, 7, and 13;
  * product_name_length, product_name_length, product_description_length, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm: impute by median, as nulls are numerical values ​​without a clear logical meaning.

In [ ]:
olist_df['products'] = olist_df['products'].dropna(subset=['product_category_name'])

numeric_cols = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]
for col in numeric_cols:
    median = olist_df['products'][col].median()
    olist_df['products'][col] = olist_df['products'][col].fillna(median)

- 3.2. Data types: Mapping the columns.

In [ ]:
dtype_products = {
    'product_id': VARCHAR(32),
    'product_category_name': VARCHAR(100),
    'product_name_lenght': Float(),
    'product_description_lenght': Float(),
    'product_photos_qty': Float(),
    'product_weight_g': Float(),
    'product_length_cm': Float(),
    'product_height_cm': Float(),
    'product_width_cm': Float()
}

- 3.3. Creating columns: Not necessary;
- 3.4. Column names: Already standardized;
- 3.5. Duplicate columns: None.

Note: There are records of ‘product_category_name’ that do not exist in the ‘category_translations’ table. MySQL does not allow creating the foreign key (FK) in ‘products’ if there is a single ‘phantom product’ in the products table, therefore, it was necessary to remove these rows.

In [ ]:
olist_df['products'] = olist_df['products'][
    olist_df['products']['product_category_name'].isin(olist_df['category_translation']['product_category_name'])
]

4. olist_sellers_dataset.

- 4.1. Data types: Mapping the columns.

In [ ]:
dtype_sellers = {
    'seller_id': VARCHAR(32),
    'seller_zip_code_prefix': Integer(),
    'seller_city': VARCHAR(100),
    'seller_state': CHAR(2)
}

- 4.2. Null values: None;
- 4.3. Create columns: Not necessary;
- 4.4. Column names: Already standardized;
- 4.5. Duplicate columns: None.

Note: seller_city has inconsistencies (such as: 'lages - sc' and 'rio de janeiro, rio de janeiro, brasil'), therefore it is necessary to remove special characters, extra spaces and standardize everything to lowercase to ensure consistency.

In [40]:
olist_df['sellers']['seller_city'] = (
    olist_df['sellers']['seller_city']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
)

5. product_category_name_translation.

- 5.1. Data types: Mapping the columns.

In [ ]:
dtype_category_translation = {
    'product_category_name': VARCHAR(100),
    'product_category_name_english': VARCHAR(100)
}

- 5.2. Null values: None;
- 5.3. Create columns: Not necessary;
- 5.4. Column names: Already standardized;
- 5.5. Duplicate columns: None.

6. olist_order_items_dataset.

- 6.1. Data types: 
  * Shipping limit date: Convert from str to datetime; 
  * Mapping the columns.

In [ ]:
olist_df['order_items']['shipping_limit_date'] = pd.to_datetime(olist_df['order_items']['shipping_limit_date'])

dtype_order_items = {
    'order_id': VARCHAR(32),
    'order_item_id': Integer(),
    'product_id': VARCHAR(32),
    'seller_id': VARCHAR(32),
    'shipping_limit_date': DateTime(),
    'price': Float(),
    'freight_value': Float()
}

- 6.2. Create columns:
  * total_item_value: Total value paid per item including shipping, useful for business questions 4 and 7.

In [43]:
olist_df['order_items']['total_item_value'] = olist_df['order_items']['price'] + olist_df['order_items']['freight_value']

- 6.3. Null values: None;
- 6.4. Column names: Already standardized;
- 6.5. Duplicate columns: None.

Note: There are product_id records that do not exist in the products table. MySQL will not allow the creation of a foreign key (FK) if there is a single "phantom product" in the order items table; therefore, such rows must be removed.

In [ ]:
olist_df['order_items'] = olist_df['order_items'][
    olist_df['order_items']['product_id'].isin(olist_df['products']['product_id'])
]

7. olist_order_payments_dataset.

- 7.1. Data Types: Mapping the Columns.

In [ ]:
dtype_order_payments = {
    'order_id': VARCHAR(32),
    'payment_sequential': Integer(),
    'payment_type': VARCHAR(20),
    'payment_installments': Integer(),
    'payment_value': Float()
}

- 7.2. Null values: none;
- 7.3. Create columns: Not necessary;
- 7.4. Column names: Already standardized;
- 7.5. Duplicate columns: None.

8. olist_order_reviews_dataset.

- 8.1. Null values:
  * review_comment_title: None of the 16 business questions focus on the textual content of the titles. Therefore, fill in with 'Sem título';
  * review_comment_message: None of the 16 business questions focus on the textual content of the messages. Therefore, fill in with 'Sem mensagem'.

In [ ]:
olist_df['order_reviews']['review_comment_title'] = olist_df['order_reviews']['review_comment_title'].fillna('Sem título')
olist_df['order_reviews']['review_comment_message'] = olist_df['order_reviews']['review_comment_message'].fillna('Sem mensagem')

- 8.2. Data types: 
  * review creation date: Convert from str to datetime;
  * review_answer_timestamp: Convert from str to datetime; * Mapping the columns.

In [ ]:
olist_df['order_reviews']['review_creation_date'] = pd.to_datetime(olist_df['order_reviews']['review_creation_date'])
olist_df['order_reviews']['review_answer_timestamp'] = pd.to_datetime(olist_df['order_reviews']['review_answer_timestamp'])

dtype_order_reviews = {
    'review_id': VARCHAR(32),
    'order_id': VARCHAR(32),
    'review_score': Integer(),
    'review_comment_title': VARCHAR(100),
    'review_comment_message': TEXT(),
    'review_creation_date': DateTime(),
    'review_answer_timestamp': DateTime()
}

- 8.3. Creating columns: Not necessary;
- 8.4. Column names: Already standardized;
- 8.5. Duplicate columns: None.

Note: There are duplicate values ​​in the review_id column, and since it will be a Primary Key, it is necessary to remove the duplicate values.

In [ ]:
olist_df['order_reviews'] = olist_df['order_reviews'].drop_duplicates(subset='review_id')

9. olist_order_customers_dataset

- 9.1. Data Types: Mapping the Columns.

In [ ]:
dtype_customers = {
    'customer_id': VARCHAR(32),
    'customer_unique_id': VARCHAR(32),
    'customer_zip_code_prefix': Integer(),
    'customer_city': VARCHAR(100),
    'customer_state': CHAR(2)
}

- 9.2. Null values: none;
- 9.3. Create columns: Not necessary;
- 9.4. Column names: Already standardized;
- 9.5. Duplicate columns: None.

Note: customer_city has inconsistencies, therefore it is necessary to remove special characters, extra spaces, and standardize everything to lowercase to ensure consistency.

In [51]:
olist_df['customers']['customer_city'] = (
    olist_df['customers']['customer_city']
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
)

10. Exporting the CSV files.

In [52]:
for name, df in olist_df.items():
    df.to_csv(f'../data/processed/{name}.csv', index=False)

11. Connecting to MySQL.

In [ ]:
username = "root"
password = "" 
host = "localhost"
port = "3306"
database = "olist_brazilian_ecommerce"
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

map_dtypes = {
    'customers': dtype_customers,
    'order_items': dtype_order_items,
    'order_payments': dtype_order_payments,
    'order_reviews': dtype_order_reviews,
    'orders': dtype_orders,
    'products': dtype_products,
    'sellers': dtype_sellers,
    'category_translation': dtype_category_translation
}

for name, df in olist_df.items():
    df.to_sql(
        name=name, 
        con=engine, 
        if_exists="replace", 
        index=False, 
        dtype=map_dtypes[name]
    )